# Exploratory Data Analysis (EDA) of Audio Signals and Features

In this notebook, we perform a deep dive into the audio signals from the IEMOCAP dataset. 
Our goal is to understand the physical and statistical properties of speech across **5 emotion classes**:
* **Angry**
* **Happy**
* **Sad**
* **Neutral**
* **Frustrated**

We will analyze the dataset balance, the duration of recordings, visual characteristics of the signal over time (Waveform, Spectrogram, Pitch, and Energy contours), and the statistical distributions of extracted acoustic features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from datasets import load_dataset
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.family': 'sans-serif'})

# Define consistent colors for the 5 target emotions
EMO_COLORS = {
    "angry": "#e74c3c", 
    "happy": "#f39c12", 
    "neutral": "#3498db", 
    "sad": "#9b59b6", 
    "frustrated": "#c0392b"
}
TARGET_EMOTIONS = list(EMO_COLORS.keys())

print("Libraries loaded successfully!")

## 1. Dataset Overview & Class Distribution
Let's load the raw HuggingFace dataset and see how many samples we have for each emotion. A balanced dataset is crucial for unbiased model training.

In [ ]:
print("Loading IEMOCAP dataset...")
ds = load_dataset("AbstractTTS/IEMOCAP", split="train")

# Count emotions
emotion_counts = {emo: 0 for emo in TARGET_EMOTIONS}
for item in ds:
    emo = item.get("major_emotion", "")
    if emo in TARGET_EMOTIONS:
        emotion_counts[emo] += 1

# Plot distribution
plt.figure(figsize=(10, 5))
sns.barplot(x=list(emotion_counts.keys()), y=list(emotion_counts.values()), palette=EMO_COLORS, order=TARGET_EMOTIONS)
plt.title("Distribution of Target Emotions in IEMOCAP", fontweight="bold", fontsize=14)
plt.xlabel("Emotion")
plt.ylabel("Number of Samples")
for i, v in enumerate(TARGET_EMOTIONS):
    plt.text(i, emotion_counts[v] + 20, str(emotion_counts[v]), ha='center', fontweight="bold")
plt.show()

## 2. Recording Durations Analysis
Do different emotions result in different speaking lengths? Are frustrated sentences usually longer than neutral ones? Let's extract the length of each recording and analyze the durations.

In [ ]:
durations = []
labels = []

# We will sample the first 2000 items to save processing time for duration extraction
# (If you want the full dataset, remove the slicing [:2000] and add tqdm)
print("Extracting audio durations...")
for item in tqdm(ds.select(range(min(2500, len(ds)))), desc="Processing audio lengths"):
    emo = item.get("major_emotion", "")
    if emo in TARGET_EMOTIONS:
        y = item["audio"]["array"]
        sr = item["audio"]["sampling_rate"]
        durations.append(len(y) / sr)
        labels.append(emo)

df_durations = pd.DataFrame({"emotion": labels, "duration_sec": durations})

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogram of durations
sns.histplot(data=df_durations, x="duration_sec", hue="emotion", kde=True, palette=EMO_COLORS, ax=axes[0], alpha=0.5)
axes[0].set_title("Distribution of Audio Durations (seconds)", fontweight="bold")
axes[0].set_xlabel("Duration (sec)")

# Boxplot to compare medians
sns.boxplot(data=df_durations, x="emotion", y="duration_sec", palette=EMO_COLORS, order=TARGET_EMOTIONS, ax=axes[1])
axes[1].set_title("Audio Duration by Emotion", fontweight="bold")
axes[1].set_xlabel("Emotion")
axes[1].set_ylabel("Duration (sec)")

plt.tight_layout()
plt.show()

print("Mean duration by emotion:")
print(df_durations.groupby("emotion")["duration_sec"].mean().round(2))

## 3. Deep Signal Analysis: Contours Over Time
Instead of just looking at the average value of pitch or energy, let's analyze the **trajectory** of the signal over time. 
We will plot the **Pitch (Fundamental Frequency - F0)** and the **Energy (RMS)** envelopes for one representative sample of each emotion.

In [ ]:
# Find one sample per emotion
samples_by_emo = {}
for item in ds:
    emo = item.get("major_emotion", "")
    if emo in TARGET_EMOTIONS and emo not in samples_by_emo:
        samples_by_emo[emo] = item
    if len(samples_by_emo) == len(TARGET_EMOTIONS): break

fig, axes = plt.subplots(len(TARGET_EMOTIONS), 2, figsize=(16, 18))

for row, emo in enumerate(TARGET_EMOTIONS):
    item = samples_by_emo[emo]
    y_ = np.array(item["audio"]["array"], dtype=np.float32)
    sr_ = item["audio"]["sampling_rate"]
    color = EMO_COLORS.get(emo, "#555")
    
    # Calculate RMS Energy over time
    rms = librosa.feature.rms(y=y_)[0]
    times_rms = librosa.frames_to_time(np.arange(len(rms)), sr=sr_)
    
    # Calculate Pitch (F0) over time
    f0, voiced_flag, voiced_probs = librosa.pyin(y_, fmin=50, fmax=500, sr=sr_)
    times_f0 = librosa.frames_to_time(np.arange(len(f0)), sr=sr_)
    
    # Plot RMS Energy Contour
    axes[row, 0].plot(times_rms, rms, color=color, linewidth=2)
    axes[row, 0].fill_between(times_rms, rms, color=color, alpha=0.3)
    axes[row, 0].set_title(f"{emo.upper()} - RMS Energy Contour", fontweight="bold")
    axes[row, 0].set_ylabel("Energy")
    axes[row, 0].set_ylim(0, max(0.1, np.max(rms) * 1.2)) # Scale decently
    
    # Plot Pitch Contour (excluding NaNs where no pitch is detected)
    axes[row, 1].plot(times_f0, f0, 'o', color=color, markersize=3, label="F0")
    axes[row, 1].set_title(f"{emo.upper()} - Pitch (F0) Contour", fontweight="bold")
    axes[row, 1].set_ylabel("Frequency (Hz)")
    axes[row, 1].set_ylim(50, 450)

plt.tight_layout()
plt.show()

### Signal Analysis Insights:
*   **Angry / Frustrated**: Notice the sharp peaks in the RMS Energy contour, showing explosive bursts of volume. The pitch points are often higher and more scattered.
*   **Sad**: The energy contour is very flat and low. The pitch (F0) stays mostly steady in the lower frequency range (often around 100-150Hz) with very little variance.
*   **Happy**: Energy shows rhythmic bounces. Pitch points move up and down significantly, showing the melody of a happy voice.

## 4. Visual Comparison: Waveform & Mel-Spectrogram
To complement the contours, here is the raw Waveform and the Mel-Spectrogram (frequency heatmaps).

In [ ]:
fig, axes = plt.subplots(len(TARGET_EMOTIONS), 2, figsize=(16, 16))
for row, emo in enumerate(TARGET_EMOTIONS):
    item = samples_by_emo[emo]
    y_ = np.array(item["audio"]["array"], dtype=np.float32)
    sr_ = item["audio"]["sampling_rate"]
    color = EMO_COLORS.get(emo, "#555")
    
    # Plot Waveform
    librosa.display.waveshow(y_, sr=sr_, ax=axes[row, 0], color=color, alpha=0.75)
    axes[row, 0].set_title(f"{emo.upper()} - Waveform", fontweight="bold")
    axes[row, 0].set_ylabel("Amplitude")
    
    # Plot Mel Spectrogram
    S = librosa.feature.melspectrogram(y=y_, sr=sr_, n_mels=64)
    S_db = librosa.power_to_db(S, ref=np.max)
    img = librosa.display.specshow(S_db, sr=sr_, x_axis="time", y_axis="mel", ax=axes[row, 1], cmap="magma")
    axes[row, 1].set_title(f"{emo.upper()} - Mel Spectrogram", fontweight="bold")
    fig.colorbar(img, ax=axes[row, 1], format="%+2.0f dB")

plt.tight_layout()
plt.show()

## 5. Acoustic Features Distributions
Let's summarize these signals across the entire dataset using Box Plots. This confirms if the patterns we saw in the single samples hold true statistically for all recordings.

In [ ]:
try:
    df_features = pd.read_csv("../data/audio_features.csv")
    df_features = df_features[df_features['emotion'].isin(TARGET_EMOTIONS)].dropna()
    
    features_to_plot = ['rms_mean', 'pitch_mean', 'zcr_mean', 'sc_mean']
    titles = ['RMS Energy (Mean Loudness)', 'Pitch (Mean F0)', 'Zero Crossing Rate (Noisiness)', 'Spectral Centroid (Brightness)']

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()

    for i, (feat, title) in enumerate(zip(features_to_plot, titles)):
        if feat in df_features.columns:
            sns.boxplot(x='emotion', y=feat, data=df_features, ax=axes[i], 
                        palette=EMO_COLORS, order=TARGET_EMOTIONS)
            axes[i].set_title(title, fontweight="bold", fontsize=14)
            axes[i].set_xlabel("Emotion")
            axes[i].set_ylabel("Feature Value")

    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print("Error: audio_features.csv not found. Cannot plot feature distributions.")

## 6. Feature Correlation Matrix
Finally, we check for multicollinearity among the features.

In [ ]:
if 'df_features' in locals() and not df_features.empty:
    selected_features = [col for col in df_features.columns if ('mean' in col) and ('mfcc' not in col)]
    selected_features += ['mfcc_1_mean', 'mfcc_2_mean'] 
    selected_features = [col for col in selected_features if col in df_features.columns]
    
    if selected_features:
        corr_matrix = df_features[selected_features].corr()
        
        plt.figure(figsize=(10, 8))
        sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
        plt.title("Correlation Matrix of Selected Audio Features", fontweight="bold", fontsize=14)
        plt.show()
    else:
        print("No relevant features found for correlation.")

## Summary & Conclusions
1.  **Class Balance**: We analyzed the number of samples per emotion, providing a baseline for our classification models.
2.  **Duration Analysis**: We checked if certain emotions are naturally longer or shorter.
3.  **Signal Trajectories**: By analyzing Pitch and Energy contours over time, we saw the dynamic nature of speech (e.g., sharp energy spikes in anger vs. flat energy in sadness).
4.  **Statistical Validation**: Box plots confirmed that these signal differences are statistically significant across the dataset, validating the use of these acoustic features.